# PumpWatch exploratory analysis
Real Binance data and independently published onset labels. Training-only exploration; held-out results are reported separately. Run from this project folder.

In [1]:
from pathlib import Path
import json
import pandas as pd
import plotly.express as px
root = Path.cwd()
if not (root / 'artifacts').exists():
    root = root.parent
d = pd.read_csv(root / 'artifacts/predictions.csv.gz')
report = json.loads((root / 'artifacts/report.json').read_text())
pd.DataFrame(report['splits']).T

,rows,positives,events,start,end
train,491554,561,187,2017-12-27 01:00:00+00:00,2019-11-14 23:59:00+00:00
validation,167309,189,63,2019-11-21 01:01:00+00:00,2020-06-21 23:59:00+00:00
test,163288,189,63,2020-06-24 01:01:00+00:00,2021-01-18 23:59:00+00:00


## Class balance and feature distributions
Synthetic test fixtures never enter this analysis. Normal means no known event. Excluded neighborhoods have target -1.

In [2]:
d.groupby(['split','target']).size().rename('windows').to_frame()

windows
split      target         
test       -1         7441
            0       163099
            1          189
train      -1        22131
            0       490993
            1          561
validation -1         7468
            0       167120
            1          189

In [3]:
training = d[(d.split == 'train') & (d.target >= 0)]
training.groupby('target')[['return_1m','volume_z','volatility_15m','trade_z']].median()

,return_1m,volume_z,volatility_15m,trade_z
target,,,,
0,0.00000,-0.273875,0.001834,-0.317077
1,0.01108,3.958369,0.023359,4.986967


In [4]:
sample = pd.concat([training.query('target == 1'), training.query('target == 0').sample(3000, random_state=42)])
px.box(sample, x='target', y='volume_z', color='target', points=False, title='Training volume anomalies')

## Held-out comparison
The model and thresholds were selected before final test evaluation.

In [5]:
pd.read_csv(root / 'artifacts/model_comparison.csv')

,Model,precision,recall,f1,average_precision,roc_auc
0,Logistic Regression,0.693333,0.825397,0.753623,0.870786,0.999476
1,Random Forest,0.791045,0.841270,0.815385,0.877313,0.999775
2,SVM,0.152738,0.280423,0.197761,0.147013,0.994341
3,Gradient Boosting,0.826733,0.883598,0.854220,0.907493,0.999818


In [6]:
pd.read_csv(root / 'artifacts/event_results.csv')

,event_id,pair,onset,eligible_minutes,detected,delay_seconds
0,event_250,EVXBTC,2020-06-24 15:59:10+00:00,3,True,50.0
1,event_104,OAXBTC,2020-06-28 16:00:00+00:00,3,True,60.0
2,event_101,GVTBTC,2020-07-04 16:00:00+00:00,3,True,60.0
3,event_098,ADXBTC,2020-07-07 16:00:00+00:00,3,True,60.0
4,event_334,CTXCBTC,2020-07-10 17:00:00+00:00,3,True,60.0
...,...,...,...,...,...,...
58,event_051,ONGBTC,2021-01-11 16:00:00+00:00,3,True,60.0
59,event_117,SNGLSBTC,2021-01-11 21:00:00+00:00,3,True,60.0
60,event_332,APPCBTC,2021-01-13 21:00:00+00:00,3,True,60.0
61,event_118,NXSBTC,2021-01-15 17:00:00+00:00,3,True,60.0


## Interpretation
Scores are uncalibrated. Event-conditioned sampling, historical regimes and uncertain labels limit generalization. See Research_Report.html and artifacts/errors.csv for the error analysis.